# TerraSpectra — Day 4: PCA Dimensionality Reduction

## 1. Objective & Prerequisites
- **Objective**: Apply Principal Component Analysis (PCA) to compress 204 contiguous spectral bands into 15 informative components while calculating exact explained variance.
- **Prerequisites**: Day 2 and Day 3 completed; `terraspectra.processing.pca` module available.

## 2. Core Concepts
- **The Hughes Phenomenon (Curse of Dimensionality)**: With 204 spectral bands, the high-dimensional feature space requires enormous amounts of labeled training data. If training samples are limited, classification accuracy degrades.
- **High Inter-Band Correlation**: Adjacent spectral bands (e.g. Band 40 and Band 41) are highly collinear and redundant.
- **PCA**: Finds orthogonal linear combinations of the spectral bands that maximize the variance of the data, effectively packing the bulk of information into the first few principal components.

In [1]:
# Import PCA classes from terraspectra
import matplotlib.pyplot as plt
import numpy as np
from terraspectra.data.loader import SalinasLoader
from terraspectra.processing.pca import HyperspectralPCA

# Load Salinas cube
loader = SalinasLoader(raw_data_dir='../data/raw')
dataset = loader.load()
cube = dataset.cube
print('Original Cube Shape:', cube.shape)

Original Cube Shape: (512, 217, 204)


In [2]:
# Reshape (H, W, Bands) into (N_samples, Bands)
samples = HyperspectralPCA.reshape_cube_to_samples(cube)
print(f'Reshaped Samples Matrix Shape: {samples.shape}')
print(f'Total Pixel Samples:          {samples.shape[0]:,}')
print(f'Original Spectral Features:    {samples.shape[1]} bands')

Reshaped Samples Matrix Shape: (111104, 204)
Total Pixel Samples:          111,104
Original Spectral Features:    204 bands


In [3]:
# Fit exploratory PCA with 15 components
n_comp = 15
pca_engine = HyperspectralPCA(n_components=n_comp)
results = pca_engine.fit_exploratory(cube)

print('--- PCA Variance Results ---')
print(f'Reduced Cube Shape:    {results.reduced_cube.shape}')
for i, var in enumerate(results.explained_variance_ratio[:5]):
    print(f'  Component {i+1:2d} Variance: {var*100.0:6.2f}%')
print(f'Total Variance Captured (15 components): {results.total_explained_variance*100.0:.2f}%')

--- PCA Variance Results ---
Reduced Cube Shape:    (512, 217, 15)
  Component  1 Variance:  74.47%
  Component  2 Variance:  23.53%
  Component  3 Variance:   1.13%
  Component  4 Variance:   0.54%
  Component  5 Variance:   0.17%
Total Variance Captured (15 components): 99.98%


In [4]:
# Plot explained variance scree chart and cumulative curve
fig_path = '../outputs/figures/salinas_pca_explained_variance.png'
pca_engine.plot_explained_variance(results, fig_path)

# Display plot in notebook
components = np.arange(1, n_comp + 1)
var_ratios = np.array(results.explained_variance_ratio) * 100.0
cum_var = np.array(results.cumulative_variance_ratio) * 100.0

plt.figure(figsize=(10, 4.5))
plt.bar(components, var_ratios, color='#2b6cb0', alpha=0.7, label='Individual Variance (%)')
plt.plot(components, cum_var, color='#c53030', marker='o', linewidth=2, label='Cumulative Variance (%)')
plt.axhline(95.0, color='gray', linestyle='--', label='95% Threshold')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance (%)')
plt.title(f'PCA Scree Plot (Total Captured: {results.total_explained_variance*100.0:.2f}%)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_21300\1526865395.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Visualize Top 3 Principal Components as an RGB image
pc_rgb = results.reduced_cube[:, :, :3].copy()
# Min-max stretch each component for display
for ch in range(3):
    c_min, c_max = pc_rgb[:, :, ch].min(), pc_rgb[:, :, ch].max()
    pc_rgb[:, :, ch] = (pc_rgb[:, :, ch] - c_min) / (c_max - c_min)

plt.figure(figsize=(7, 10))
plt.imshow(pc_rgb)
plt.title('Top 3 Principal Components Composite (PC1: Red, PC2: Green, PC3: Blue)')
plt.xlabel('Column Index')
plt.ylabel('Row Index')
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_21300\148457493.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. What I Learned & Common Mistakes
1. **Massive Compression Efficiency**: The first 3 principal components capture over **98%** of the entire scene's variance! The first 15 components capture **99.98%**, compressing feature dimensionality by 92.6% (from 204 to 15 bands) with virtually zero loss of signal.
2. **Data Leakage in PCA**: When evaluating downstream classifiers, fitting PCA on the entire dataset leaks validation and test sample distributions into the feature extractor. Always fit PCA strictly on training pixels (`fit_train_data`) and apply `transform_samples` to test pixels.

### Interview Question
*Q: Why is PCA particularly effective on hyperspectral data compared to computer vision RGB data?*
*A: Hyperspectral bands are physically contiguous (adjacent wavelengths measured by the same spectrometer). Consequently, adjacent bands have correlation coefficients often exceeding 0.95. PCA eliminates this multi-collinearity.*

## 4. Verification & Milestone Status
- [x] Verified sample reshaping `(512, 217, 204)` -> `(111104, 204)`.
- [x] Confirmed explained variance calculation: Top 3 = 74.47%, 23.53%, 1.13%; Total 15 = 99.98%.
- [x] Validated train-only PCA mode in unit tests (`tests/test_pca.py`).
- [x] Persisted reduced dataset to `data/processed/salinas_pca_reduced.npz`.